In [1]:
import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging
load_dotenv(override=True)
openai = OpenAI()
MODEL = "llama3.2"

c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

response = ollama.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Hi"
        }
    ]
)

print(response.choices[0].message.content)

How can I assist you today?


### Some test data

In [3]:
test_results = ScannerAgent().test_scan()
test_results

DealSelection(deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and

## Create fake functions

We create fake functions for teaching and testing the agent workflow before connecting real systems.

In [4]:
def scan_the_internet_for_bargains() -> str:
    """ This tool scans the internet for great deals and gets a curated list of promising deals """
    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    return test_results.model_dump_json()

In [5]:
def estimate_true_value(description: str) -> str:
    """
    This tool estimates the true value of a product based on a text description of it
    """
    print(f"Fake function to estimating true value of {description[:20]}... - this always returns $300")
    return f"Product {description} has an estimated true value of $300"

In [24]:
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """
    This tool notifies the user of a great deal, given a description of it, the price of the deal, and the estimated true value
    """
    print(f"Fake function to notify user of {description} which costs {deal_price} and estimate is {estimated_true_value}")
    return "notification sent ok"

scan_the_internet_for_bargains()  ->	Scanner Agent

estimate_true_value()	    ->        Ensemble Agent (or pricing system)

notify_user_of_deal()	    ->        Messaging Agent

# Check

In [7]:
scan_the_internet_for_bargains()

Fake function to scan the internet - this returns a hardcoded set of deals


'{"deals":[{"product_description":"The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku\'s operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.","price":178.0,"url":"https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},{"product_description":"The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom contro

In [8]:
estimate_true_value("a Samsung Galaxy 23 ultra")

Fake function to estimating true value of a Samsung Galaxy 23 ... - this always returns $300


'Product a Samsung Galaxy 23 ultra has an estimated true value of $300'

In [9]:
notify_user_of_deal("Iphone 15 pro max", 500, 800, "https://www.apple.com/iphone")

Fake function to notify user of Iphone 15 pro max which costs 500 and estimate is 800


'notification sent ok'

# Structured Outputs

### Without Structured Output

Prompt:

Find a good deal.

LLM might return:

I found a Sony Headphone deal for around $250.

You can check it at example.com.

or

Sony Headphones

Price: $250

URL: example.com

The format changes every time.

### With Structured Output

We define:

class Deal(BaseModel):

    product_description: str

    price: float

    url: str

Now the model must return:

{

  "product_description": "Sony WH-1000XM5 Wireless Headphones",

  "price": 250,

  "url": "https://example.com"

}

Same format every time.

## We always prefer JSON format

In [10]:
scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [11]:
tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
 ]

In [12]:
tools

[{'type': 'function',
  'function': {'name': 'scan_the_internet_for_bargains',
   'description': 'Returns top bargains scraped from the internet along with the price each item is being offered for',
   'parameters': {'type': 'object',
    'properties': {},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'estimate_true_value',
   'description': 'Given the description of an item, estimate how much it is actually worth',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'The description of the item to be estimated'}},
    'required': ['description'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'notify_user_of_deal',
   'description': 'Send the user a push notification about the single most compelling deal; only call this one time',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description

### Here the tools are 

1) scan_function

2) estimate_function

3) notify_function

In [25]:
def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        print(tool_call.function.arguments)
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

### It returns like 

estimate_true_value("Sony Headphones")

       ↓

     $300

       ↓

Returned to LLM

In [26]:
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

bargain means find the product with the biggest discount

In [27]:
messages

[{'role': 'system',
  'content': 'You find great deals on bargain products using your tools, and notify the user of the best bargain.'},
 {'role': 'user',
  'content': '\nFirst, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.\nThen pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.\nThen just reply OK to indicate success.\n'}]

In [35]:
done = False
while not done:
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

'The most compelling deal is the "Product Laptop with i7 processor and 16 GB RAM" priced at $446, which has an estimated true value of $300.\n\nThis means that you\'re saving $156 by purchasing this laptop during the sale.'

# Planning Agent

Remove Fake Functions

          ↓

Use Real Agents

An Autonomous Planning Agent is an AI agent that plans and executes multiple steps on its own. In this project, the fake testing functions are replaced with real Scanner, Ensemble, and Messaging Agents, allowing the agent to find real deals, estimate actual values, and send real notifications automatically.

In [38]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [39]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

In [40]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection)

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cpu
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is ready


In [ ]:
agent.plan()

## Autonomous Planning Agent

It reads:

1. Find deals

2. Estimate values

3. Pick best deal

4. Notify user